In [65]:
import sys
from pathlib import Path

# Set root
cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.features.preprocessing import load_model_data
from src.evaluation import splits
from src.evaluation.make_splits import load_split

In [ ]:
# Load the cleaned/merged dataset

X, y, meta = load_model_data()
FEATURES = list(X.columns)

data = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y))
print(data["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(data["holdout"].sum()))

# df = pd.read_csv(DATA_PATH, dtype={"gm_well_id": "string"}, parse_dates=["gm_samp_collection_date"])
# df = load_split(df)

# print(df["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(df["holdout"].sum()))

TypeError: load_model_data() got an unexpected keyword argument 'subbasin'

In [48]:
# Set the training folds and the holdout

train_val = data[~data["holdout"]].reset_index(drop=True)       # Training set with 4 folds


X_tv = train_val[FEATURES]
y_tv = train_val["log_nitrate"]

# Print the sizes of the training folds
print(train_val["fold"].value_counts().sort_index().to_dict())

{0: 1035, 1: 972, 2: 1143, 3: 1041}


Here you can begin training your model.

In [49]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.base import clone
import xgboost

In [50]:
models = {
    "dummy": DummyRegressor(strategy="mean"),
    "brt": xgboost.XGBRegressor(learning_rate=.1, max_depth=3, n_estimators=100)
}

template = models["dummy"]

oof_pred = np.full(len(train_val), np.nan)        # out-of-fold prediction for every row
fold_scores = []

for train, test in splits.iter_folds(train_val["fold"]):
    model = clone(template)
    model.fit(X_tv[train], y_tv[train])
    oof_pred[test] = model.predict(X_tv[test])

    fold = int(train_val.loc[test, "fold"].iloc[0])
    fold_scores.append({"fold": fold,
                        "rmse_log": root_mean_squared_error(y_tv[test], oof_pred[test])})

fold_scores = pd.DataFrame(fold_scores)
exceeds = np.expm1(y_tv) > 10
print(fold_scores.round(3).to_string(index=False))
print(f"Overall RMSE (log): {root_mean_squared_error(y_tv, oof_pred):.3f}")
print(f"Accuracy of the >10 mg/L call: {np.mean((np.expm1(oof_pred) > 10) == exceeds):.3f}")


 fold  rmse_log
    0     1.227
    1     1.183
    2     1.222
    3     1.314
Overall RMSE (log): 1.238
Accuracy of the >10 mg/L call: 0.662


Save this last file for the final evaluation (when we have compared all of our models and have chosen a final one).

In [51]:
# # ============================================================
# # FINAL EVALUATION ON THE HOLD-OUT FOLD
# # Run this ONCE, for the final chosen model only, after all
# # model choices (features, hyperparameters) are finished.
# # ============================================================
# RUN_FINAL_EVALUATION = False

# if RUN_FINAL_EVALUATION:
#     holdout_df = df[df["holdout"]].reset_index(drop=True)
#     final_model = ...                      # the chosen model, refit on all of train_val
#     final_model.fit(X, y)
#     holdout_pred = final_model.predict(holdout_df[FEATURES])
#     ...                                    # scores on np.log1p(holdout_df[TARGET])


# Nested CV Example

In [52]:
FEATURES

['gm_well_category',
 'soil_slopegradwta',
 'soil_aws0150wta',
 'soil_hydgrpdcd_pct_a',
 'soil_hydgrpdcd_pct_b',
 'soil_hydgrpdcd_pct_c',
 'elevation_m',
 'topographic_position',
 'pct_veg_berry_500m',
 'pct_vineyard_500m',
 'pct_orchard_500m',
 'pct_grain_field_pasture_500m',
 'pct_idle_fallow_500m',
 'pct_urban_500m',
 'pct_cropland_500m',
 'crop_cycles_500m',
 'pct_multicrop_500m',
 'outside_dwr_polygons_pct_500m',
 'pct_orchard_1000m',
 'pct_grain_field_pasture_1000m',
 'pct_idle_fallow_1000m',
 'pct_urban_1000m',
 'outside_dwr_polygons_pct_1000m',
 'delta_veg_berry_pp_500m',
 'delta_vineyard_pp_500m',
 'delta_orchard_pp_500m',
 'delta_grain_field_pasture_pp_500m',
 'delta_idle_fallow_pp_500m',
 'delta_cropland_pp_500m',
 'delta_multicrop_pp_500m',
 'delta_dwr_polygon_footprint_pp_500m',
 'delta_veg_berry_pp_1000m',
 'delta_vineyard_pp_1000m',
 'delta_orchard_pp_1000m',
 'delta_grain_field_pasture_pp_1000m',
 'delta_idle_fallow_pp_1000m',
 'delta_urban_pp_1000m',
 'delta_cropland_p

In [53]:
SUBFOLD_SEED = 77
MODEL_SEED = 99
N_SUBFOLDS = 4
THRESHOLD_MG_L = 10

In [54]:
from src.models import random_forest
# "data" comes from load_split (see earlier template)
# this function creates the subfolds

development, X_dev, y_dev = random_forest.make_inner_subfolds(data, FEATURES, N_SUBFOLDS = N_SUBFOLDS, THRESHOLD_MG_L = THRESHOLD_MG_L, SEED = SUBFOLD_SEED)

subfold    0    1    2    3
fold                       
0        187  195  237  416
1        109  194  230  439
2        408  258  302  175
3        186  160  224  471


In [ ]:
param_grid = {
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [50, 100, 200, 500],
    "max_depth": [3, 4, 5],
    "min_child_weight": [1, 5, 20],
    "subsample": [0.7, 1.0],
}


In [56]:
# # customize this function to your estimator. Leave all other fields as is. 
# from sklearn.model_selection import GridSearchCV
# import xgboost

# def make_search_brt(inner_cv, param_grid, SEED):
#     return GridSearchCV(
#         estimator=xgboost.XGBRegressor(
#             objective="reg:squarederror",
#             random_state=SEED,
#             n_jobs=1,
#         ),
#         param_grid=param_grid,
#         scoring="neg_root_mean_squared_error",
#         cv=inner_cv,
#         refit=True,
#         n_jobs=-1,
#     )


In [57]:
from sklearn.model_selection import RandomizedSearchCV

def make_search_brt(inner_cv, param_grid, SEED):
    return RandomizedSearchCV(
        estimator=xgboost.XGBRegressor(objective="reg:squarederror", random_state=SEED, n_jobs=1),
        param_distributions=param_grid,
        n_iter=40,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        refit=True,
        random_state=SEED,
        n_jobs=-1,
    )


In [58]:
#call the function fit_model_with_nested_cv

outer_results, best_params_by_fold = (
random_forest.fit_model_with_nested_cv(
        development,
        X_dev,
        y_dev,
        param_grid,
        make_search_brt,
        N_SUBFOLDS=N_SUBFOLDS,
        THRESHOLD_MG_L=THRESHOLD_MG_L,
        SEED=MODEL_SEED,
        PRINT=True
    )
)

Nested CV results (outer and inner fold RMSE):
 outer_fold  rmse_log  best_inner_rmse_log  inner_fold_0_rmse_log  inner_fold_1_rmse_log  inner_fold_2_rmse_log  inner_fold_3_rmse_log
          0     1.118                1.132                  1.093                  1.101                  1.066                  1.268
          1     1.375                1.067                  1.043                  1.151                  0.839                  1.232
          2     1.129                1.085                  0.860                  1.043                  1.092                  1.345
          3     1.135                1.119                  1.100                  1.150                  0.964                  1.261

Best parameters per outer fold:
 outer_fold  subsample  n_estimators  min_child_weight  max_depth
          0        1.0           100                 1          5
          1        1.0           100                20          3
          2        1.0           100           

In [59]:
def find_best_hyperparameter(all_best_params, ignore=("outer_fold", "seed")):
    param_cols = [c for c in all_best_params.columns if c not in ignore]
    overall = (all_best_params
               .groupby(param_cols, dropna=False)
               .size()
               .reset_index(name="count")
               .sort_values("count", ascending=False))
    overall["frequency"] = overall["count"] / len(all_best_params)
    return overall


In [60]:
runs = []
for seed in [1, 2, 3]:
    outer_results, best_params_by_fold = random_forest.fit_model_with_nested_cv(
        development, X_dev, y_dev, param_grid, make_search_brt,
        N_SUBFOLDS=N_SUBFOLDS, THRESHOLD_MG_L=THRESHOLD_MG_L, SEED=seed, PRINT=False,
    )
    runs.append(best_params_by_fold.assign(seed=seed))

all_best_params = pd.concat(runs, ignore_index=True)       # 3 runs × 4 outer folds = 12 rows


In [61]:
frequency = find_best_hyperparameter(all_best_params)
frequency


,subsample,n_estimators,min_child_weight,max_depth,count,frequency
3,1.0,100,20,3,3,0.250000
4,1.0,100,20,4,3,0.250000
1,1.0,100,1,3,2,0.166667
2,1.0,100,1,5,1,0.083333
0,0.7,100,1,4,1,0.083333
5,1.0,200,1,5,1,0.083333
6,1.0,200,20,4,1,0.083333


In [62]:
param_cols = [c for c in all_best_params.columns if c not in ("outer_fold", "seed")]
best_params = frequency[param_cols].iloc[[0]].to_dict("records")[0]
best_params


{'subsample': 1.0, 'n_estimators': 100, 'min_child_weight': 20, 'max_depth': 3}

In [64]:
# Normal CV function (just outer folds)

model = xgboost.XGBRegressor(learning_rate=.1, max_depth=3, n_estimators=100)
random_forest.fit_model_with_normal_cv(development, X_dev, y_dev, model, THRESHOLD_MG_L = 10)

CV results (outer fold RMSE):
 outer_fold  rmse_log
          0     1.073
          1     1.299
          2     1.093
          3     1.060
Overall RMSE (log): 1.132
Accuracy of the >10 mg/L call: 0.695
